# 01 - Build the training data

Runs the data pipeline on Colab: **download -> clean -> LaBSE filter -> train tokenizer**. The output (line-aligned `*.hi`/`*.en` splits + a SentencePiece model) is what `02_train` trains on.

**Before running:** (1) push your latest commits from the Mac so the clone/pull below is current; (2) set a **GPU runtime** (Runtime -> Change runtime type -> GPU) for the LaBSE step.

**Persistence:** Colab's `/content` is wiped on reset and the LaBSE pass is slow, so the config cell **mounts Google Drive** and writes everything under `DATA_ROOT` on Drive. After this one full build, every step skips itself on later runs (its outputs already exist on Drive) and `02_train` reads straight from Drive. Run the cells top to bottom.

## 1. Get the repo onto Colab + install

The cell below **clones** the repo onto Colab (or **pulls** the latest if it's already there). Push from the Mac first, or this brings stale code. (Committing/pushing is done on the **Mac**, not here.)

In [1]:
import os, sys

REPO_URL = "https://github.com/Rishi-Jain-27/hindi-translator.git"
REPO_DIR = "/content/hindi-translator"

# clone if missing, else pull the latest
!test -d {REPO_DIR} && git -C {REPO_DIR} pull || git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("src"))  # import nmt straight from src/ (robust to editable-install state)
print("cwd:", os.getcwd())

remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 9 (delta 6), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 1.24 KiB | 423.00 KiB/s, done.
From https://github.com/Rishi-Jain-27/hindi-translator
   eeccb85..148cb9f  main       -> origin/main
Updating eeccb85..148cb9f
Fast-forward
 notebooks/01_data.ipynb | 25 ++++++++-----------------
 pyproject.toml          |  5 +++--
 src/nmt/data/clean.py   | 18 +++++++++---------
 3 files changed, 20 insertions(+), 28 deletions(-)
cwd: /content/hindi-translator


In [2]:
# all deps come from pyproject (datasets, sentence-transformers, indic-nlp-library, sentencepiece, py3langid, ...)
!pip install -e .

Obtaining file:///content/hindi-translator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 746.1/746.1 kB 16.4 MB/s eta 0:00:00a 0:00:01
  Building editable for nmt (pyproject.toml) ... done
  Created wheel for nmt: filename=nmt-0.1.0-0.editable-py3-none-any.whl size=1288 sha256=83ed3abb7297fb4435fe8f97158cb03188ce9c7829437ece46e597386e0f2293
  Stored in directory: /tmp/pip-ephem-wheel-cache-_gp9ad_r/wheels/bd/c2/ef/ae818ff943d77ea8d63ef07aea61a1b82808716362dc169d4c
Successfully built nmt
  Attempting uninstall: nmt
    Found existing installation: nmt 0.1.0
    Uninstalling nmt-0.1.0:
      Successfully uninstalled nmt-0.1.0


## 2. Config (paths)

In [ ]:
from nmt.config import DataConfig
from google.colab import drive

# Persist raw + cache + tokenizer to Google Drive so they survive a runtime reset.
# Colab's /content is wiped on disconnect and the LaBSE pass is slow to redo. On Drive,
# every step below skips itself on later runs (its outputs already exist) -- so this is the
# last full rebuild. The first mount pops up a Drive authorization prompt.
drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/hindi-translator/data"   # raw + cache live here, on Drive

cfg = DataConfig(raw_dir=f"{DATA_ROOT}/raw", cache_dir=f"{DATA_ROOT}/cache")
print("raw_dir:", cfg.raw_dir, "| cache_dir:", cfg.cache_dir, "| vocab:", cfg.vocab_size, "| corpus:", cfg.corpus)

## 3. Download IITB

Pulls the IIT Bombay Hi-En corpus from HuggingFace and writes line-aligned `train`/`dev`/`test` `.hi`/`.en` files to `raw_dir`. Idempotent - skips splits already staged.

In [4]:
from nmt.data.download import download

raw_paths = download(cfg)            # {split: (hi_path, en_path)}
print("sample hi:", open(raw_paths["train"][0], encoding="utf-8").readline().strip())
print("sample en:", open(raw_paths["train"][1], encoding="utf-8").readline().strip())

[download] train: already staged -> data/raw/train.hi, data/raw/train.en
[download] dev: already staged -> data/raw/dev.hi, data/raw/dev.en
[download] test: already staged -> data/raw/test.hi, data/raw/test.en
sample hi: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
sample en: Give your application an accessibility workout


## 4. Clean

NFC + Indic normalization, **py3langid** language-id filtering, length-ratio gate, dedupe, and removal of train pairs that leak into dev/test. **Only train is filtered**; dev/test are normalized but never dropped. Prints the kept/dropped breakdown.

In [5]:
from nmt.data.clean import clean

clean_paths = clean(cfg)   # writes *.clean.hi/.en; prints kept/dropped stats

[clean] train: {'total': 1652981, 'langid': 490609, 'ratio': 21250, 'dup': 129419, 'kept': 1011703} -> data/cache/train.clean.hi, data/cache/train.clean.en


## 5. LaBSE filter (GPU; slow, one-time, cached)

Embeds both sides of each cleaned train pair with LaBSE and drops pairs whose cross-lingual cosine similarity is below `cfg.labse_threshold` (0.70), removing misaligned/noisy pairs. This is the long step - cached as `train.labse.*`. LaBSE is a pretrained *filter*, not part of the from-scratch model.

In [6]:
from nmt.data.labse_filter import labse_filter

labse_paths = labse_filter(cfg)     # GPU auto-used if present; writes train.labse.*; dev/test pass through

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

[labse] train: kept 726389/1011703 (thr=0.7, mean_sim=0.749) -> data/cache/train.labse.hi, data/cache/train.labse.en


## 6. Train the tokenizer

One shared SentencePiece **unigram 16k** vocab over both languages on the LaBSE-filtered train, with `byte_fallback` (no `<unk>`) and the fixed id contract pad/unk/bos/eos = 0/1/2/3. Writes `spm_unigram_16000.model`.

In [7]:
from nmt.data.tokenizer import train_tokenizer

tok = train_tokenizer(cfg, labse_paths["train"][0], labse_paths["train"][1])
print("vocab_size:", tok.vocab_size, "| pad/unk/bos/eos:", tok.pad_id, tok.unk_id, tok.bos_id, tok.eos_id)

[tokenizer] trained unigram vocab=16000 -> data/cache/spm_unigram_16000.model
vocab_size: 16000 | pad/unk/bos/eos: 0 1 2 3


## 7. Sanity checks

In [8]:
# line counts (hi/en must match per split) + a tokenizer round-trip
def _count(p):
    with open(p, encoding="utf-8") as f:
        return sum(1 for _ in f)

for split, (h, e) in labse_paths.items():
    nh, ne = _count(h), _count(e)
    print(f"{split:5s}: {nh} hi / {ne} en  {'OK' if nh == ne else 'MISMATCH'}")

sample = open(labse_paths["train"][0], encoding="utf-8").readline().strip()
ids = tok.encode(sample, add_eos=True)
print("\nsample :", sample)
print("ids    :", ids[:20], "...")
print("decoded:", tok.decode(ids))
print("has <unk>?", tok.unk_id in ids, "(should be False - byte_fallback)")

train: 726389 hi / 726389 en  OK
dev  : 520 hi / 520 en  OK
test : 2507 hi / 2507 en  OK

sample : अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
ids    : [334, 3677, 277, 1832, 4527, 343, 13508, 278, 1080, 1609, 3] ...
decoded: अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
has <unk>? False (should be False - byte_fallback)
